# Phase 2 — Data Cleaning & Validation

**Project:** Australian Stock Market Analytics, Valuation & Machine Learning

This notebook starts from the original Excel dataset and creates a separate cleaning DataFrame. The raw Excel file is never modified.

**Important:** KMD is kept and flagged until its duplicate records are manually reviewed.

## Expected project structure

```text
australian-stock-market-data-science/
├── data/
│   ├── raw/
│   │   └── Aus 1st Half 2025 (1)(1).xlsx
│   ├── processed/
│   └── metadata/
├── notebooks/
│   ├── 01_data_understanding.ipynb
│   └── 02_data_cleaning.ipynb
├── src/
├── models/
└── dashboard/
```

Run this notebook from the `notebooks/` folder.

In [1]:
# ============================================
# CELL 1 — IMPORT LIBRARIES
# ============================================

import os
import re
import warnings

import numpy as np
import pandas as pd
import openpyxl

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
# ============================================
# CELL 2 — DEFINE PROJECT PATHS
# ============================================

DATA_PATH = "E:\Projects\Australian Stock Market Data Science Project\data\Aus 1st Half 2025.xlsx"
PROCESSED_DIR = "../data/processed"
METADATA_DIR = "../data/metadata"

os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(METADATA_DIR, exist_ok=True)

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"Dataset not found at: {DATA_PATH}")

print("Raw dataset:", DATA_PATH)
print("Processed directory:", PROCESSED_DIR)
print("Metadata directory:", METADATA_DIR)

Raw dataset: E:\Projects\Australian Stock Market Data Science Project\data\Aus 1st Half 2025.xlsx
Processed directory: ../data/processed
Metadata directory: ../data/metadata


In [3]:
# ============================================
# CELL 3 — LOAD ORIGINAL EXCEL DATASET
# ============================================

SHEET_NAME = "Sheet1"

df_raw = pd.read_excel(DATA_PATH, sheet_name=SHEET_NAME)

print("Raw dataset loaded successfully.")
print("Shape:", df_raw.shape)

Raw dataset loaded successfully.
Shape: (54, 16)


In [4]:
# ============================================
# CELL 4 — VERIFY RAW DATASET
# ============================================

print("Rows:", len(df_raw))
print("Columns:", len(df_raw.columns))

print("\nOriginal columns:")
for i, column in enumerate(df_raw.columns, start=1):
    print(f"{i:02d}. {column}")

print("\nFirst 5 rows:")
display(df_raw.head())

Rows: 54
Columns: 16

Original columns:
01. TY
02. Company name
03. TOTAL SHARES
04. MARKET CAPITAL(ml)
05. Total Shareholders Equity(ml)
06. 2H2024(JUL-DEC)
07. 1H2025(JAN-JUNE)
08. EPS IN LAST 12 MONTHS
09. EPS AP 24/25
10. P/E BASED ON LAST ANNUAL REPORT
11. AVERAGE P/E FOR LAST 10 YEARS
12. P/E BASED ON LAST 12 MONTHS
13. PRICE
14. book value per share $
15. 52 Weeks Low - High
16. BUY PRICE

First 5 rows:


,TY,Company name,TOTAL SHARES,MARKET CAPITAL(ml),Total Shareholders Equity(ml),2H2024(JUL-DEC),1H2025(JAN-JUNE),EPS IN LAST 12 MONTHS,EPS AP 24/25,P/E BASED ON LAST ANNUAL REPORT,AVERAGE P/E FOR LAST 10 YEARS,P/E BASED ON LAST 12 MONTHS,PRICE,book value per share $,52 Weeks Low - High,BUY PRICE
0,VEA,Viva Energy Group Ltd,1625651926,"3,511.4082",NaN,-0.3000,12.3000,12.0000,4.8000,450.0000,NaN,18.0000,2.1600,1.0270,1.41 - 2.83,NaN
1,DDR,Dicker Data Ltd,180817855,"1,871.4648",NaN,23.9700,21.7900,45.7600,43.6200,237.2765,NaN,22.6180,10.3500,1.3890,7.58 - 10.94,NaN
2,FMG,Fortescue Metals Group Ltd,3078964918,"68,075.9143",NaN,50.5000,39.5500,90.0500,109.7000,201.5497,NaN,24.5530,22.1100,9.8740,13.18 - 22.11,NaN
3,RIO,RIO Tinto Ltd,371216214,"51,402.3092",NaN,315.2000,296.0000,611.2000,669.5000,206.8260,NaN,22.6554,138.4700,54.4740,100.75 - 140.58,NaN
4,APE,Eagers Automotive Ltd,282364338,"7,578.6588",NaN,34.8000,46.4000,81.2000,80.2000,334.6633,NaN,33.0542,26.8400,4.6360,10.90 - 35.64,NaN


In [5]:
# ============================================
# CELL 5 — CREATE CLEANING COPY
# ============================================

# Keep the raw dataset untouched.
df_clean = df_raw.copy()

print("Raw shape:", df_raw.shape)
print("Working shape:", df_clean.shape)

Raw shape: (54, 16)
Working shape: (54, 16)


## 1. Standardize Column Names

The original Excel names remain available in `df_raw`. The working DataFrame uses Python-friendly names.

In [6]:
# ============================================
# CELL 6 — COLUMN NAME MAPPING
# ============================================

column_mapping = {
    "TY": "ticker",
    "Company name": "company_name",
    "TOTAL SHARES": "total_shares",
    "MARKET CAPITAL(ml)": "market_cap_m",
    "Total Shareholders Equity(ml)": "shareholders_equity_m",
    "2H2024(JUL-DEC)": "eps_2h2024",
    "1H2025(JAN-JUNE)": "eps_1h2025",
    "EPS IN LAST 12 MONTHS": "eps_ltm",
    "EPS AP 24/25": "eps_ap_24_25",
    "P/E BASED ON LAST ANNUAL REPORT": "pe_annual_report",
    "AVERAGE P/E FOR LAST 10 YEARS": "pe_10y_average",
    "P/E BASED ON LAST 12 MONTHS": "pe_ltm",
    "PRICE": "price",
    "book value per share $": "book_value_per_share",
    "52 Weeks Low - High": "week52_range",
    "BUY PRICE": "buy_price"
}

missing_mappings = set(df_clean.columns) - set(column_mapping.keys())
if missing_mappings:
    raise ValueError(f"Columns without mappings: {missing_mappings}")

print("All columns have mappings.")

All columns have mappings.


In [7]:
# ============================================
# CELL 7 — RENAME COLUMNS
# ============================================

df_clean = df_clean.rename(columns=column_mapping)
print(df_clean.columns.tolist())

['ticker', 'company_name', 'total_shares', 'market_cap_m', 'shareholders_equity_m', 'eps_2h2024', 'eps_1h2025', 'eps_ltm', 'eps_ap_24_25', 'pe_annual_report', 'pe_10y_average', 'pe_ltm', 'price', 'book_value_per_share', 'week52_range', 'buy_price']


## 2. Investigate Duplicate Tickers

Phase 1 found 54 records but only 53 unique tickers. The duplicate ticker is KMD. We inspect it but do not automatically delete either record.

In [8]:
# ============================================
# CELL 8 — IDENTIFY DUPLICATE TICKERS
# ============================================

duplicate_ticker_mask = df_clean["ticker"].duplicated(keep=False)
duplicate_ticker_rows = df_clean.loc[duplicate_ticker_mask].sort_values("ticker")

display(duplicate_ticker_rows[["ticker", "company_name"]])

,ticker,company_name
20,KMD,KMD Brands Limited
44,KMD,KMD Brands Ltd


In [9]:
# ============================================
# CELL 9 — DETAILED KMD COMPARISON
# ============================================

kmd_records = df_clean[df_clean["ticker"].astype(str).str.upper().eq("KMD")].copy()
display(kmd_records.T)

,20,44
ticker,KMD,KMD
company_name,KMD Brands Limited,KMD Brands Ltd
total_shares,711667484,711667484
market_cap_m,170.8002,170.8002
shareholders_equity_m,NaN,NaN
eps_2h2024,3.0000,3.0000
eps_1h2025,10.4000,10.4000
eps_ltm,13.4000,13.4000
eps_ap_24_25,13.4000,13.4000
pe_annual_report,17.9104,17.9104


In [10]:
# ============================================
# CELL 10 — FLAG DUPLICATE TICKERS
# ============================================

df_clean["duplicate_ticker_flag"] = df_clean["ticker"].duplicated(keep=False)

print("Rows with duplicate tickers:", int(df_clean["duplicate_ticker_flag"].sum()))
display(df_clean.loc[df_clean["duplicate_ticker_flag"], ["ticker", "company_name", "duplicate_ticker_flag"]])

Rows with duplicate tickers: 2


,ticker,company_name,duplicate_ticker_flag
20,KMD,KMD Brands Limited,True
44,KMD,KMD Brands Ltd,True


### Duplicate-ticker checkpoint

Do not delete KMD automatically. Review Cell 9 first.

## 3. Handle Completely Empty Columns

Phase 1 established three completely empty fields: `shareholders_equity_m`, `pe_10y_average`, and `buy_price`. They are excluded from the analytical dataset while the raw Excel file remains untouched.

In [11]:
# ============================================
# CELL 11 — IDENTIFY EMPTY COLUMNS
# ============================================

empty_columns = [column for column in df_clean.columns if df_clean[column].isna().all()]

empty_column_audit = pd.DataFrame({
    "Column": empty_columns,
    "Missing Count": [df_clean[column].isna().sum() for column in empty_columns],
    "Missing Percentage": [df_clean[column].isna().mean() * 100 for column in empty_columns],
    "Action": ["Exclude from analytical dataset" for column in empty_columns]
})

display(empty_column_audit)

,Column,Missing Count,Missing Percentage,Action
0,shareholders_equity_m,54,100.0000,Exclude from analytical dataset
1,pe_10y_average,54,100.0000,Exclude from analytical dataset
2,buy_price,54,100.0000,Exclude from analytical dataset


In [12]:
# ============================================
# CELL 12 — REMOVE EMPTY COLUMNS
# ============================================

# Remove only from df_clean; df_raw is unchanged.
df_clean = df_clean.drop(columns=empty_columns)

print("Removed:")
for column in empty_columns:
    print("-", column)
print("\nWorking shape:", df_clean.shape)

Removed:
- shareholders_equity_m
- pe_10y_average
- buy_price

Working shape: (54, 14)


In [13]:
# ============================================
# CELL 13 — CHECK REMAINING MISSING VALUES
# ============================================

missing_summary = pd.DataFrame({
    "Missing Count": df_clean.isna().sum(),
    "Missing Percentage": (df_clean.isna().mean() * 100).round(2)
})

display(missing_summary[missing_summary["Missing Count"] > 0])

,Missing Count,Missing Percentage
pe_annual_report,2,3.7000


In [14]:
# ============================================
# CELL 14 — FLAG MISSING ANNUAL P/E
# ============================================

# Do not impute broken-source P/E values.
df_clean["annual_pe_missing"] = df_clean["pe_annual_report"].isna()

display(df_clean.loc[df_clean["annual_pe_missing"], ["ticker", "company_name", "pe_annual_report"]])

,ticker,company_name,pe_annual_report
12,LAU,Lindsay Australia Limited,NaN
36,PTL,Pental Ltd,NaN


## 4. Enforce Numeric Data Types

In [15]:
# ============================================
# CELL 15 — EXPECTED NUMERIC COLUMNS
# ============================================

expected_numeric_columns = [
    "total_shares", "market_cap_m", "eps_2h2024", "eps_1h2025",
    "eps_ltm", "eps_ap_24_25", "pe_annual_report", "pe_ltm",
    "price", "book_value_per_share"
]

print(*expected_numeric_columns, sep="\n")

total_shares
market_cap_m
eps_2h2024
eps_1h2025
eps_ltm
eps_ap_24_25
pe_annual_report
pe_ltm
price
book_value_per_share


In [16]:
# ============================================
# CELL 16 — SAFE NUMERIC CONVERSION
# ============================================

for column in expected_numeric_columns:
    df_clean[column] = pd.to_numeric(df_clean[column], errors="coerce")

print("Numeric conversion completed.")
display(df_clean[expected_numeric_columns].dtypes)

Numeric conversion completed.


total_shares              int64
market_cap_m            float64
eps_2h2024              float64
eps_1h2025              float64
eps_ltm                 float64
eps_ap_24_25            float64
pe_annual_report        float64
pe_ltm                  float64
price                   float64
book_value_per_share    float64
dtype: object

In [17]:
# ============================================
# CELL 17 — VALIDATE TOTAL SHARES
# ============================================

invalid_shares = df_clean[df_clean["total_shares"].isna() | (df_clean["total_shares"] <= 0)]
print("Invalid share-count records:", len(invalid_shares))
display(invalid_shares[["ticker", "company_name", "total_shares"]])

Invalid share-count records: 0


,ticker,company_name,total_shares


In [18]:
# ============================================
# CELL 18 — VALIDATE CURRENT PRICE
# ============================================

invalid_prices = df_clean[df_clean["price"].isna() | (df_clean["price"] <= 0)]
print("Invalid price records:", len(invalid_prices))
display(invalid_prices[["ticker", "company_name", "price"]])

Invalid price records: 0


,ticker,company_name,price


In [19]:
# ============================================
# CELL 19 — CHECK INFINITE VALUES
# ============================================

numeric_columns = df_clean.select_dtypes(include=np.number).columns.tolist()

infinite_summary = pd.DataFrame({
    "Positive Infinity": np.isposinf(df_clean[numeric_columns]).sum(),
    "Negative Infinity": np.isneginf(df_clean[numeric_columns]).sum()
})

display(infinite_summary)

,Positive Infinity,Negative Infinity
total_shares,0,0
market_cap_m,0,0
eps_2h2024,0,0
eps_1h2025,0,0
eps_ltm,0,0
eps_ap_24_25,0,0
pe_annual_report,0,0
pe_ltm,0,0
price,0,0
book_value_per_share,0,0


## 5. Parse and Validate the 52-Week Price Range

In [20]:
# ============================================
# CELL 20 — DEFINE 52-WEEK RANGE PARSER
# ============================================

def parse_52_week_range(value):
    # Parse text such as "1.50 - 3.20" into low and high.
    if pd.isna(value):
        return pd.Series([np.nan, np.nan])

    text = str(value).strip()
    pattern = r"^\s*([0-9]+(?:\.[0-9]+)?)\s*-\s*([0-9]+(?:\.[0-9]+)?)\s*$"
    match = re.match(pattern, text)

    if not match:
        return pd.Series([np.nan, np.nan])

    return pd.Series([float(match.group(1)), float(match.group(2))])

In [21]:
# ============================================
# CELL 21 — PARSE 52-WEEK RANGE
# ============================================

df_clean[["week52_low", "week52_high"]] = df_clean["week52_range"].apply(parse_52_week_range)

display(df_clean[["ticker", "price", "week52_range", "week52_low", "week52_high"]].head(15))

,ticker,price,week52_range,week52_low,week52_high
0,VEA,2.1600,1.41 - 2.83,1.4100,2.8300
1,DDR,10.3500,7.58 - 10.94,7.5800,10.9400
2,FMG,22.1100,13.18 - 22.11,13.1800,22.1100
3,RIO,138.4700,100.75 - 140.58,100.7500,140.5800
4,APE,26.8400,10.90 - 35.64,10.9000,35.6400
5,AFG,2.2500,1.425 - 2.92,1.4250,2.9200
6,ASH,0.2150,0.128 - 0.245,0.1280,0.2450
7,ASG,4.2300,1.605 - 4.70,1.6050,4.7000
8,SGM,17.7700,11.63 - 18.11,11.6300,18.1100
9,ACL,2.8400,2.38 - 3.83,2.3800,3.8300


In [22]:
# ============================================
# CELL 22 — VALIDATE 52-WEEK VALUES
# ============================================

invalid_52_week = df_clean[
    df_clean["week52_low"].isna()
    | df_clean["week52_high"].isna()
    | (df_clean["week52_low"] <= 0)
    | (df_clean["week52_high"] <= 0)
    | (df_clean["week52_low"] >= df_clean["week52_high"])
]

print("Invalid 52-week ranges:", len(invalid_52_week))
display(invalid_52_week[["ticker", "week52_range", "week52_low", "week52_high"]])

Invalid 52-week ranges: 1


,ticker,week52_range,week52_low,week52_high
37,WOW,25.52 0- 33.76,NaN,NaN


In [23]:
# ============================================
# CELL 23 — CHECK PRICE VS 52-WEEK RANGE
# ============================================

price_outside_52_week_range = df_clean[
    (df_clean["price"] < df_clean["week52_low"])
    | (df_clean["price"] > df_clean["week52_high"])
]

print("Prices outside reported 52-week range:", len(price_outside_52_week_range))
display(price_outside_52_week_range[["ticker", "price", "week52_low", "week52_high"]])

Prices outside reported 52-week range: 0


,ticker,price,week52_low,week52_high


## 6. Recalculate Market Capitalization

The source workbook contains incorrect formula references for WOW and JBH. We independently calculate market capitalization from total shares and price.

In [24]:
# ============================================
# CELL 24 — RECALCULATE MARKET CAPITALIZATION
# ============================================

df_clean["market_cap_m_recalculated"] = (
    df_clean["total_shares"] * df_clean["price"] / 1_000_000
)

print("Market capitalization recalculated.")

Market capitalization recalculated.


In [25]:
# ============================================
# CELL 25 — AUDIT MARKET CAPITALIZATION
# ============================================

market_cap_audit = df_clean[[
    "ticker", "company_name", "market_cap_m", "market_cap_m_recalculated"
]].copy()

market_cap_audit["difference"] = (
    market_cap_audit["market_cap_m"] - market_cap_audit["market_cap_m_recalculated"]
)
market_cap_audit["absolute_difference"] = market_cap_audit["difference"].abs()

display(market_cap_audit.sort_values("absolute_difference", ascending=False).head(10))

,ticker,company_name,market_cap_m,market_cap_m_recalculated,difference,absolute_difference
37,WOW,Woolworths Group Ltd,"3,053.9933","35,902.7452","-32,848.7519","32,848.7519"
43,JBH,JB Hi-Fi Ltd,"3,031.8313","9,775.5512","-6,743.7199","6,743.7199"
2,FMG,Fortescue Metals Group Ltd,"68,075.9143","68,075.9143",0.0000,0.0000
3,RIO,RIO Tinto Ltd,"51,402.3092","51,402.3092",0.0000,0.0000
0,VEA,Viva Energy Group Ltd,"3,511.4082","3,511.4082",0.0000,0.0000
1,DDR,Dicker Data Ltd,"1,871.4648","1,871.4648",0.0000,0.0000
6,ASH,Ashley Services Group,30.9548,30.9548,0.0000,0.0000
7,ASG,Autosports Group Ltd,857.3504,857.3504,0.0000,0.0000
8,SGM,Sims Ltd,"3,433.7232","3,433.7232",0.0000,0.0000
9,ACL,Australian Clinical Labs Ltd,550.5593,550.5593,0.0000,0.0000


In [26]:
# ============================================
# CELL 26 — USE VALIDATED MARKET CAP
# ============================================

# Preserve the original Excel value for auditability.
df_clean["market_cap_m_original"] = df_clean["market_cap_m"]

# Use the independently calculated value for analysis.
df_clean["market_cap_m"] = df_clean["market_cap_m_recalculated"]

df_clean = df_clean.drop(columns=["market_cap_m_recalculated"])

print("Validated market capitalization is now in 'market_cap_m'.")

Validated market capitalization is now in 'market_cap_m'.


## 7. Recalculate LTM EPS

Phase 1 showed that LTM EPS is consistent with `2H2024 + 1H2025`. We reproduce the calculation independently.

In [27]:
# ============================================
# CELL 27 — RECALCULATE LTM EPS
# ============================================

df_clean["eps_ltm_recalculated"] = (
    df_clean["eps_2h2024"] + df_clean["eps_1h2025"]
)

df_clean["eps_ltm_difference"] = (
    df_clean["eps_ltm"] - df_clean["eps_ltm_recalculated"]
)

eps_audit = df_clean[[
    "ticker", "eps_ltm", "eps_ltm_recalculated", "eps_ltm_difference"
]].copy()

display(eps_audit)

,ticker,eps_ltm,eps_ltm_recalculated,eps_ltm_difference
0,VEA,12.0000,12.0000,0.0000
1,DDR,45.7600,45.7600,0.0000
2,FMG,90.0500,90.0500,0.0000
3,RIO,611.2000,611.2000,0.0000
4,APE,81.2000,81.2000,0.0000
5,AFG,15.1000,15.1000,0.0000
6,ASH,1.5100,1.5100,0.0000
7,ASG,16.2600,16.2600,0.0000
8,SGM,9.9000,9.9000,0.0000
9,ACL,17.1500,17.1500,0.0000


In [28]:
# ============================================
# CELL 28 — IDENTIFY LTM EPS PROBLEMS
# ============================================

eps_problems = eps_audit[
    eps_audit["eps_ltm_difference"].abs() > 0.0001
]

print("LTM EPS calculation problems:", len(eps_problems))
display(eps_problems)

LTM EPS calculation problems: 0


,ticker,eps_ltm,eps_ltm_recalculated,eps_ltm_difference


In [29]:
# ============================================
# CELL 29 — USE VALIDATED LTM EPS
# ============================================

df_clean["eps_ltm_original"] = df_clean["eps_ltm"]
df_clean["eps_ltm"] = df_clean["eps_ltm_recalculated"]

df_clean = df_clean.drop(columns=["eps_ltm_recalculated", "eps_ltm_difference"])

print("Validated LTM EPS is now in 'eps_ltm'.")

Validated LTM EPS is now in 'eps_ltm'.


## 8. Recalculate LTM P/E

Based on the source workbook units, the validated formula is `PRICE / LTM EPS × 100`. This corrects the known WOW and JBH errors.

In [30]:
# ============================================
# CELL 30 — RECALCULATE LTM P/E
# ============================================

df_clean["pe_ltm_recalculated"] = np.where(
    df_clean["eps_ltm"] != 0,
    (df_clean["price"] / df_clean["eps_ltm"]) * 100,
    np.nan
)

print("LTM P/E recalculated.")

LTM P/E recalculated.


In [31]:
# ============================================
# CELL 31 — AUDIT LTM P/E
# ============================================

pe_audit = df_clean[[
    "ticker", "pe_ltm", "pe_ltm_recalculated"
]].copy()

pe_audit["difference"] = (
    pe_audit["pe_ltm"] - pe_audit["pe_ltm_recalculated"]
)
pe_audit["absolute_difference"] = pe_audit["difference"].abs()

display(pe_audit.sort_values("absolute_difference", ascending=False).head(10))

,ticker,pe_ltm,pe_ltm_recalculated,difference,absolute_difference
37,WOW,3.1686,37.2497,-34.0811,34.0811
43,JBH,6.5556,21.1371,-14.5816,14.5816
2,FMG,24.5530,24.5530,0.0000,0.0000
3,RIO,22.6554,22.6554,0.0000,0.0000
0,VEA,18.0000,18.0000,0.0000,0.0000
1,DDR,22.6180,22.6180,0.0000,0.0000
6,ASH,14.2384,14.2384,0.0000,0.0000
7,ASG,26.0148,26.0148,0.0000,0.0000
8,SGM,179.4949,179.4949,0.0000,0.0000
9,ACL,16.5598,16.5598,0.0000,0.0000


In [32]:
# ============================================
# CELL 32 — IDENTIFY LTM P/E PROBLEMS
# ============================================

pe_problems = pe_audit[
    pe_audit["absolute_difference"] > 0.01
]

print("LTM P/E calculation problems:", len(pe_problems))
display(pe_problems)

LTM P/E calculation problems: 2


,ticker,pe_ltm,pe_ltm_recalculated,difference,absolute_difference
37,WOW,3.1686,37.2497,-34.0811,34.0811
43,JBH,6.5556,21.1371,-14.5816,14.5816


In [33]:
# ============================================
# CELL 33 — USE VALIDATED LTM P/E
# ============================================

df_clean["pe_ltm_original"] = df_clean["pe_ltm"]
df_clean["pe_ltm"] = df_clean["pe_ltm_recalculated"]

df_clean = df_clean.drop(columns=["pe_ltm_recalculated"])

print("Validated LTM P/E is now in 'pe_ltm'.")

Validated LTM P/E is now in 'pe_ltm'.


## 9. Annual P/E Handling

The annual-report P/E field contains two missing values because the source workbook contains broken `#REF!` formulas. We do not impute these values.

In [34]:
# ============================================
# CELL 34 — ANNUAL P/E MISSINGNESS
# ============================================

df_clean["annual_pe_missing"] = df_clean["pe_annual_report"].isna()
annual_pe_missing = df_clean[df_clean["annual_pe_missing"]]

print("Missing annual P/E records:", len(annual_pe_missing))
display(annual_pe_missing[["ticker", "company_name", "pe_annual_report"]])

Missing annual P/E records: 2


,ticker,company_name,pe_annual_report
12,LAU,Lindsay Australia Limited,NaN
36,PTL,Pental Ltd,NaN


## 10. Negative EPS / P-E Analysis

Negative earnings and EPS can be legitimate financial observations, so they are investigated rather than automatically removed.

In [35]:
# ============================================
# CELL 35 — IDENTIFY NEGATIVE EPS
# ============================================

negative_eps = df_clean[
    (df_clean["eps_2h2024"] < 0)
    | (df_clean["eps_1h2025"] < 0)
    | (df_clean["eps_ltm"] < 0)
]

print("Companies with at least one negative EPS measure:", len(negative_eps))
display(negative_eps[[
    "ticker", "company_name", "eps_2h2024", "eps_1h2025", "eps_ltm", "pe_ltm"
]])

Companies with at least one negative EPS measure: 7


,ticker,company_name,eps_2h2024,eps_1h2025,eps_ltm,pe_ltm
0,VEA,Viva Energy Group Ltd,-0.3000,12.3000,12.0000,18.0000
10,ORA,Orora Limited,67.7000,-56.3000,11.4000,18.8596
15,GNC,Graincorp Ltd,26.2000,-8.0000,18.2000,45.9341
29,BEN,Bendigo and Adelaide Bank Ltd,38.3000,-21.1000,17.2000,60.5233
42,NTD,National Tyre & Wheel Ltd,28.8400,-1.1100,27.7300,0.9196
45,CFK,Collins Foods Ltd,20.5000,-13.0000,7.5000,138.9333
52,ALD,Ampol Ltd,-47.3000,10.6000,-36.7000,-88.4741


In [36]:
# ============================================
# CELL 36 — NEGATIVE FINANCIAL VALUE SUMMARY
# ============================================

financial_columns_for_negative_check = [
    "eps_2h2024", "eps_1h2025", "eps_ltm",
    "eps_ap_24_25", "pe_ltm", "pe_annual_report"
]

negative_summary = pd.DataFrame({
    "Negative Count": [
        (df_clean[column] < 0).sum()
        for column in financial_columns_for_negative_check
    ]
}, index=financial_columns_for_negative_check)

display(negative_summary)

,Negative Count
eps_2h2024,2
eps_1h2025,5
eps_ltm,1
eps_ap_24_25,0
pe_ltm,1
pe_annual_report,0


## 11. Outlier Investigation

Financial outliers are flagged, not automatically deleted. Extreme values may represent legitimate differences between companies.

In [37]:
# ============================================
# CELL 37 — IQR OUTLIER ANALYSIS
# ============================================

outlier_columns = [
    "total_shares", "market_cap_m", "eps_ltm",
    "eps_ap_24_25", "pe_ltm", "price", "book_value_per_share"
]

outlier_summary = []

for column in outlier_columns:
    q1 = df_clean[column].quantile(0.25)
    q3 = df_clean[column].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    outlier_mask = (
        (df_clean[column] < lower_bound)
        | (df_clean[column] > upper_bound)
    )

    outlier_summary.append({
        "Column": column,
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "Lower Bound": lower_bound,
        "Upper Bound": upper_bound,
        "Outlier Count": int(outlier_mask.sum())
    })

outlier_summary_df = pd.DataFrame(outlier_summary)
display(outlier_summary_df)

,Column,Q1,Q3,IQR,Lower Bound,Upper Bound,Outlier Count
0,total_shares,"186,489,108.7500","1,199,841,181.7500","1,013,352,073.0000","-1,333,539,000.7500","2,719,869,291.2500",6
1,market_cap_m,551.1922,"17,024.2157","16,473.0235","-24,158.3431","41,733.7509",8
2,eps_ltm,11.5500,95.9750,84.4250,-115.0875,222.6125,6
3,eps_ap_24_25,11.9000,97.2500,85.3500,-116.1250,225.2750,6
4,pe_ltm,11.9972,27.9668,15.9696,-11.9571,51.9211,7
5,price,2.1825,22.0425,19.8600,-27.6075,51.8325,5
6,book_value_per_share,1.0445,9.5705,8.5260,-11.7445,22.3595,4


In [38]:
# ============================================
# CELL 38 — CREATE OUTLIER FLAGS
# ============================================

for column in outlier_columns:
    q1 = df_clean[column].quantile(0.25)
    q3 = df_clean[column].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    df_clean[f"{column}_outlier"] = (
        (df_clean[column] < lower_bound)
        | (df_clean[column] > upper_bound)
    )

print("Outlier flags created successfully.")

Outlier flags created successfully.


## 12. Final Data-Quality Checks

In [39]:
# ============================================
# CELL 39 — FINAL MISSING-VALUE CHECK
# ============================================

final_missing = pd.DataFrame({
    "Missing Count": df_clean.isna().sum(),
    "Missing Percentage": (df_clean.isna().mean() * 100).round(2)
})

display(final_missing[final_missing["Missing Count"] > 0])

,Missing Count,Missing Percentage
pe_annual_report,2,3.7000
week52_low,1,1.8500
week52_high,1,1.8500


In [40]:
# ============================================
# CELL 40 — FINAL DUPLICATE CHECK
# ============================================

print("Duplicate rows:", int(df_clean.duplicated().sum()))
print("Duplicate ticker rows:", int(df_clean["ticker"].duplicated().sum()))

print("\nDuplicate ticker records:")
display(df_clean.loc[
    df_clean["ticker"].duplicated(keep=False),
    ["ticker", "company_name"]
])

Duplicate rows: 0
Duplicate ticker rows: 1

Duplicate ticker records:


,ticker,company_name
20,KMD,KMD Brands Limited
44,KMD,KMD Brands Ltd


In [41]:
# ============================================
# CELL 41 — FINAL NUMERIC VALIDATION
# ============================================

numeric_columns_final = df_clean.select_dtypes(include=np.number).columns.tolist()

positive_infinity_count = np.isposinf(df_clean[numeric_columns_final]).sum().sum()
negative_infinity_count = np.isneginf(df_clean[numeric_columns_final]).sum().sum()

print("Positive infinity values:", int(positive_infinity_count))
print("Negative infinity values:", int(negative_infinity_count))

Positive infinity values: 0
Negative infinity values: 0


In [42]:
# ============================================
# CELL 42 — FINAL DATASET OVERVIEW
# ============================================

print("=" * 70)
print("PHASE 2 — CLEAN DATASET OVERVIEW")
print("=" * 70)
print(f"\nRows: {len(df_clean)}")
print(f"Columns: {len(df_clean.columns)}")
print(f"Unique companies: {df_clean['company_name'].nunique()}")
print(f"Unique tickers: {df_clean['ticker'].nunique()}")
print(f"Duplicate rows: {df_clean.duplicated().sum()}")
print(f"Duplicate ticker rows: {df_clean['ticker'].duplicated().sum()}")
print(f"Missing annual P/E records: {df_clean['annual_pe_missing'].sum()}")
print(f"Invalid 52-week ranges: {len(invalid_52_week)}")
print(f"Prices outside reported 52-week range: {len(price_outside_52_week_range)}")
print(f"LTM EPS validation problems: {len(eps_problems)}")
print(f"LTM P/E validation problems: {len(pe_problems)}")

PHASE 2 — CLEAN DATASET OVERVIEW

Rows: 54
Columns: 27
Unique companies: 54
Unique tickers: 53
Duplicate rows: 0
Duplicate ticker rows: 1
Missing annual P/E records: 2
Invalid 52-week ranges: 1
Prices outside reported 52-week range: 0
LTM EPS validation problems: 0
LTM P/E validation problems: 2


## 13. Create Cleaning Audit

In [43]:
# ============================================
# CELL 43 — CREATE CLEANING AUDIT
# ============================================

cleaning_audit = pd.DataFrame([
    {
        "Issue": "Completely empty columns",
        "Action": "Excluded from analytical dataset",
        "Details": ", ".join(empty_columns)
    },
    {
        "Issue": "Duplicate ticker",
        "Action": "Flagged; not automatically removed",
        "Details": "KMD requires manual/source verification"
    },
    {
        "Issue": "Annual P/E #REF!/missing values",
        "Action": "Retained as missing; no imputation",
        "Details": "Underlying source formula was broken"
    },
    {
        "Issue": "Market capitalization",
        "Action": "Recalculated from total shares × price",
        "Details": "Corrects source formula-reference errors"
    },
    {
        "Issue": "LTM EPS",
        "Action": "Recalculated from 2H2024 + 1H2025",
        "Details": "Independent validation of source calculation"
    },
    {
        "Issue": "LTM P/E",
        "Action": "Recalculated from price / LTM EPS × 100",
        "Details": "Corrects inconsistent source values"
    },
    {
        "Issue": "52-week range",
        "Action": "Parsed and numerically validated",
        "Details": "Original text field retained"
    },
    {
        "Issue": "Negative EPS/P-E",
        "Action": "Retained",
        "Details": "May represent legitimate negative earnings"
    },
    {
        "Issue": "Financial outliers",
        "Action": "Flagged, not removed",
        "Details": "Extreme financial values can be legitimate"
    }
])

display(cleaning_audit)

,Issue,Action,Details
0,Completely empty columns,Excluded from analytical dataset,"shareholders_equity_m, pe_10y_average, buy_price"
1,Duplicate ticker,Flagged; not automatically removed,KMD requires manual/source verification
2,Annual P/E #REF!/missing values,Retained as missing; no imputation,Underlying source formula was broken
3,Market capitalization,Recalculated from total shares × price,Corrects source formula-reference errors
4,LTM EPS,Recalculated from 2H2024 + 1H2025,Independent validation of source calculation
5,LTM P/E,Recalculated from price / LTM EPS × 100,Corrects inconsistent source values
6,52-week range,Parsed and numerically validated,Original text field retained
7,Negative EPS/P-E,Retained,May represent legitimate negative earnings
8,Financial outliers,"Flagged, not removed",Extreme financial values can be legitimate


## 14. Save Processed Dataset and Audit Reports

In [44]:
# ============================================
# CELL 44 — SAVE CLEAN DATASET
# ============================================

clean_csv_path = os.path.join(PROCESSED_DIR, "australian_stocks_cleaned.csv")
df_clean.to_csv(clean_csv_path, index=False)

print("Clean dataset saved:")
print(clean_csv_path)

Clean dataset saved:
../data/processed\australian_stocks_cleaned.csv


In [45]:
# ============================================
# CELL 45 — SAVE AUDIT REPORTS
# ============================================

cleaning_audit.to_csv(
    os.path.join(PROCESSED_DIR, "data_cleaning_audit.csv"),
    index=False
)

market_cap_audit.to_csv(
    os.path.join(PROCESSED_DIR, "market_cap_validation.csv"),
    index=False
)

eps_audit.to_csv(
    os.path.join(PROCESSED_DIR, "eps_validation.csv"),
    index=False
)

pe_audit.to_csv(
    os.path.join(PROCESSED_DIR, "pe_validation.csv"),
    index=False
)

empty_column_audit.to_csv(
    os.path.join(PROCESSED_DIR, "empty_column_audit.csv"),
    index=False
)

print("Audit reports saved successfully.")

Audit reports saved successfully.


In [46]:
# ============================================
# CELL 46 — SAVE PHASE 2 SUMMARY
# ============================================

phase2_summary = {
    "Original rows": len(df_raw),
    "Cleaned rows": len(df_clean),
    "Original columns": len(df_raw.columns),
    "Cleaned columns": len(df_clean.columns),
    "Unique companies": df_clean["company_name"].nunique(),
    "Unique tickers": df_clean["ticker"].nunique(),
    "Duplicate rows": int(df_clean.duplicated().sum()),
    "Duplicate ticker rows": int(df_clean["ticker"].duplicated().sum()),
    "Completely empty columns removed": len(empty_columns),
    "Missing annual P/E records": int(df_clean["annual_pe_missing"].sum()),
    "Invalid 52-week ranges": len(invalid_52_week),
    "Prices outside reported 52-week range": len(price_outside_52_week_range),
    "LTM EPS validation problems": len(eps_problems),
    "LTM P/E validation problems": len(pe_problems)
}

phase2_summary_df = pd.DataFrame(
    phase2_summary.items(),
    columns=["Metric", "Value"]
)

phase2_summary_df.to_csv(
    os.path.join(PROCESSED_DIR, "phase2_summary.csv"),
    index=False
)

display(phase2_summary_df)

,Metric,Value
0,Original rows,54
1,Cleaned rows,54
2,Original columns,16
3,Cleaned columns,27
4,Unique companies,54
5,Unique tickers,53
6,Duplicate rows,0
7,Duplicate ticker rows,1
8,Completely empty columns removed,3
9,Missing annual P/E records,2


In [50]:
# ==========================================
# CELL 47 — REMOVE DUPLICATE TICKER
# ==========================================

# Remove duplicate ticker records.
# Keep the first occurrence of each ticker.

df_clean = df_clean.drop_duplicates(
    subset=["ticker"],
    keep="first"
).copy()

print("Shape after removing duplicate ticker:")
print(df_clean.shape)

print("\nDuplicate tickers remaining:")
print(df_clean["ticker"].duplicated().sum())

Shape after removing duplicate ticker:
(53, 27)

Duplicate tickers remaining:
0


In [51]:
# ==========================================
# CELL 48 — NORMALIZE COMPANY NAMES
# ==========================================

df_clean["company_name"] = (
    df_clean["company_name"]
    .astype(str)
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

print("Company names normalized successfully.")

Company names normalized successfully.


In [52]:
# ==========================================
# CELL 49 — 52-WEEK RANGE QUALITY FLAG
# ==========================================

df_clean["week52_range_valid"] = (
    df_clean["week52_low"].notna() &
    df_clean["week52_high"].notna()
)

print("52-week range validity:")
print(df_clean["week52_range_valid"].value_counts())

print("\nCompanies with invalid 52-week range:")
print(
    df_clean.loc[
        ~df_clean["week52_range_valid"],
        ["ticker", "company_name", "week52_range"]
    ]
)

52-week range validity:
week52_range_valid
True     52
False     1
Name: count, dtype: int64

Companies with invalid 52-week range:
   ticker          company_name    week52_range
37    WOW  Woolworths Group Ltd  25.52 0- 33.76


In [54]:
# ==========================================
# CELL 50 — FINAL PHASE 2 VALIDATION
# ==========================================

print("FINAL PHASE 2 VALIDATION")
print("=" * 50)

print("Rows:", len(df_clean))
print("Unique companies:", df_clean["company_name"].nunique())
print("Unique tickers:", df_clean["ticker"].nunique())

print("\nDuplicate rows:", df_clean.duplicated().sum())
print("Duplicate tickers:", df_clean["ticker"].duplicated().sum())

print("\nRemaining missing values:")
missing_values = df_clean.isna().sum()
print(missing_values[missing_values > 0])

print("\nInvalid 52-week ranges:")
print((~df_clean["week52_range_valid"]).sum())


# ==========================================
# SAVE FINAL CLEANED DATASET
# ==========================================

# PROCESSED_DIR is currently a string,
# so use string concatenation instead of "/".

output_file = os.path.join(
    PROCESSED_DIR,
    "australian_stocks_cleaned.csv"
)

df_clean.to_csv(
    output_file,
    index=False
)

print("\nFinal cleaned dataset saved to:")
print(output_file)

FINAL PHASE 2 VALIDATION
Rows: 53
Unique companies: 53
Unique tickers: 53

Duplicate rows: 0
Duplicate tickers: 0

Remaining missing values:
pe_annual_report    2
week52_low          1
week52_high         1
dtype: int64

Invalid 52-week ranges:
1

Final cleaned dataset saved to:
../data/processed\australian_stocks_cleaned.csv


# Phase 2 Completion Checklist

- [ ] Raw Excel dataset reloaded
- [ ] Raw dataset remains untouched
- [ ] Columns standardized
- [ ] KMD duplicate inspected and flagged
- [ ] Completely empty columns excluded
- [ ] Missing annual P/E retained as missing
- [ ] 52-week range parsed and validated
- [ ] Market capitalization independently recalculated
- [ ] LTM EPS independently validated
- [ ] LTM P/E independently recalculated
- [ ] Negative EPS values investigated, not deleted
- [ ] Outliers flagged, not automatically removed
- [ ] Final missing/duplicate/infinite checks performed
- [ ] Clean dataset and audit reports saved

**Before Phase 3:** review the KMD duplicate and the outputs of `kmd_records.T`, `invalid_52_week`, `price_outside_52_week_range`, `market_cap_audit`, and `pe_audit`.